# Linear from scrach


X: draw from standard normal distribution, with (num_examples, len(w))


y: the linear result without bias


bias draw from normal(0, 0.01)


reshape y as row vector, and retrun X and y

(**我们使用线性模型参数$\mathbf{w} = [2, -3.4]^\top$、$b = 4.2$
和噪声项$\epsilon$生成数据集及其标签：

$$\mathbf{y}= \mathbf{X} \mathbf{w} + b + \mathbf\epsilon.$$
**)

In [63]:
import random
import torch

def synthetic_data(w, b, num_examples):
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.mv(X, w) + b # mm must matrix * matrix, 
    # in this part we could use matmul
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))

true_w = torch.tensor([2, -3.4])
print(true_w.shape)
true_b = torch.tensor([4.2])
features, labels = synthetic_data(true_w, true_b, 1000)
print(f"features{features[0]}, \n labels{labels[0]}")

torch.Size([2])
featurestensor([ 1.3683, -0.9504]), 
 labelstensor([10.1606])


在下面的代码中，我们定义一个data_iter函数， 该函数接收批量大小、特征矩阵和标签向量作为输入，生成大小为batch_size的小批量。 每个小批量包含一组特征和标签。

input: batch_size, features, labels

return: list of batch_sized features and labels

In [64]:
def data_iter(batch_size, features, labels):
    num_examples = len(features)
    indices = list(range(num_examples))
    random.shuffle(indices)
    for i in range(0, num_examples, batch_size):
        batch_indices = torch.tensor(indices[i:min(i+ batch_size, num_examples)])
        yield features[batch_indices], labels[batch_indices]

init parameters

In [65]:
w = torch.normal(0, 0.001, size=(2, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

Define functions and loss function and sgd function

In [66]:
def linreg(X, w, b):
    return torch.matmul(X, w) + b

def squared_loss(y_hat, y):
    return (y_hat - y.reshape(y_hat.shape)) ** 2 /2

def sgd(params, lr, batch_size):
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()


Training

In [67]:
lr = 0.03
num_epochs = 3
net = linreg
loss = squared_loss
batch_size = 10

for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        l = loss(net(X, w, b), y)
        l.sum().backward()
        sgd([w, b], lr, batch_size)
    with torch.no_grad():
        train_l = loss(net(features, w, b), labels)
        print(f'epoch {epoch + 1}, loss {float(train_l.mean()):f}')

epoch 1, loss 0.032377
epoch 2, loss 0.000115
epoch 3, loss 0.000051


# concise linear

In [68]:
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

In [69]:
true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000)

In [70]:
def load_array(data_arrays, batch_size, is_train=True):
    dataset = data.TensorDataset(*data_arrays)# transform the data array into tensor dataset
    return data.DataLoader(dataset, batch_size, is_train)

batch_size = 10
data_iter = load_array((features, labels), batch_size)

In [71]:
from torch import nn
net = nn. Sequential(nn.Linear(2, 1))
# def linreg(X, w, b):
#     return torch.matmul(X, w) + b
loss = nn.MSELoss()
# def squared_loss(y_hat, y):
#     return (y_hat - y.reshape(y_hat.shape)) ** 2 /2
trainer = torch.optim.SGD(net.parameters(), lr = 0.03)
# def sgd(params, lr, batch_size):
#     with torch.no_grad():
#         for param in params:
#             param -= lr * param.grad / batch_size
#             param.grad.zero_()

Train

In [72]:

num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X) ,y)#  l = loss(net(X, w, b), y)
        # trainer.zero_grad()
        l.backward()# l.sum().backward()
        trainer.step()# sgd([w, b], lr, batch_size)
        trainer.zero_grad()# inside sgd
    train_l = loss(net(features), labels) 
    print(f'epoch {epoch + 1}, loss {train_l:f}')


    # for X, y in data_iter(batch_size, features, labels):
    #     l = loss(net(X, w, b), y)
    #     l.sum().backward()
    #     sgd([w, b], lr, batch_size)
    # with torch.no_grad():
    #     train_l = loss(net(features, w, b), labels)

epoch 1, loss 0.000264
epoch 2, loss 0.000100
epoch 3, loss 0.000100
